In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
import os

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "."
ANNOTATIONS_PATH = os.path.join(BASE_DIR, "annotations.xml")
IMAGES_DIR = os.path.join(BASE_DIR, "images")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "crops")

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image

#identify the classes we are interested in for this dataset
classes = [
    "free_parking_space",
    "not_free_parking_space",
    "partially_free_parking_space",
]

#creates a folder for the classes identified 
for cls in classes:
    os.makedirs(os.path.join(OUTPUT_DIR, cls), exist_ok=True)

#load and pharse the XML file using the polygon data given 
tree = ET.parse(ANNOTATIONS_PATH)
root = tree.getroot()
num_crops = 0

#loop through each <image> node in the XML file 
for image_node in root.findall("image"):

    #extract the image name from the annotation
    img_name = image_node.get("name")          # e.g. "images/0.png"
    img_path = os.path.join(BASE_DIR, img_name)

    #error handling for if image is missing 
    if not os.path.isfile(img_path):
        print("Missing:", img_path)
        continue

    # loads image of the full lot 
    img = Image.open(img_path).convert("RGB")

    #loops through the <polygon> annotation in the image, with each polygon representing a parking spot 
    for i, poly in enumerate(image_node.findall("polygon")):

        #gets the class label for the polygon 
        label = poly.get("label")
        if label not in classes:
            continue

        # Parse polygon -> bounding box
        pts = [p.split(",") for p in poly.get("points").split(";")]
        pts = [(float(x), float(y)) for x, y in pts]
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]

        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        # Crop the area inside the bounding box from the full image
        crop = img.crop((xmin, ymin, xmax, ymax))

        # Build output filename: <original_image_name>_<polygon_idx>.jpg
        base_name = os.path.splitext(os.path.basename(img_name))[0]
        out_name = f"{base_name}_{i}.jpg"
        out_path = os.path.join(OUTPUT_DIR, label, out_name)

        # Save to disk
        crop.save(out_path)
        num_crops += 1

#final summary
print(f"DONE. Total crops: {num_crops}")

DONE. Total crops: 903
